In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime
import logging
import pandas as pd
import time
import csv
import sys
import os
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

## Create a directory to store log files if it doesn't exist


In [2]:
log_directory = "logs"
if not os.path.exists(log_directory):
    os.makedirs(log_directory, exist_ok = True)

* Get the name of the current script file

In [3]:
%%javascript
IPython.notebook.kernel.execute('nb_name = "' + IPython.notebook.notebook_name + '"')

<IPython.core.display.Javascript object>

In [4]:
nb_full_path = os.path.join(os.getcwd(), nb_name)
current_file_name = nb_full_path.split("/")[-1].split(".")[0]

In [5]:
# Create a timestamp for the log file name
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [6]:
# Define the log file path
log_file_name = f"{current_file_name}_scraping_{timestamp}.log"
log_file = os.path.join(log_directory, log_file_name)

In [7]:
# Configure the logging
logging.basicConfig(filename=log_file, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [8]:
def scrape_award_info(text):
    # Extract category
    category_start_index = text.find("Category:") + len("Category:")
    category_end_index = text.find("Year:")
    category = text[category_start_index:category_end_index].strip()

    # Extract year
    year_start_index = text.find("Year:") + len("Year:")
    year_end_index = [i for i in range(len(text)) if text.startswith('\n', i)][-1]
    year = text[year_start_index:year_end_index].strip()

    # Extract text below the year
    description = text[year_end_index:].strip()
    description = description.replace("â€œ", "'")
    description = description.replace("â€", "'")
    description = description.replace("\x9d", "")

    return {
        "category": category,
        "year": year,
        "description": description
    }


In [9]:
def scrape_awards_table(url):
    service=Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)
    driver = webdriver.Chrome(service=service)
    
    # Load the webpage
#     driver.maximize_window()    
    driver.get(url)

    # Get the page source after Selenium renders the JavaScript
    page_source = driver.page_source

    # Parse the HTML using BeautifulSoup
    soup = BeautifulSoup(page_source, 'html.parser')

    award_wraps = soup.find_all("div", class_="award-card__wrap")

    # List to store scraped data
    scraped_data = []
    year = ""
    county = ""
    program = ""
    program_link = ""
    category = ""
    best_in_category = False


    # Extract information from each award card
    for award_wrap in award_wraps:
        year = award_wrap.find("span", class_="award-card__year").text.strip()
        county = award_wrap.find("span", class_="award-card__county").text.strip()
        program = award_wrap.find("span", class_="award-card__program").text.strip()
        program_link = "naco.org"+award_wrap.find("span", class_="award-card__program").find("a")["href"]

        try:
            category_span = award_wrap.find("span", class_="award-card__category")
            category = category_span.text.strip().split('\n')[0].strip()
            # Check if <br> tag is present and contains "(Best In Category)"
            if "(Best In Category)" in category_span.text:
                category_variable = "Best In Category"
                best_in_category = True
            else:
                category_variable = ""

        except Exception as e:
            print(f"Error expanding row: {e}")

        # Append extracted information to the list
        scraped_data.append({
            "year": year,
            "county": county,
            "program": program,
            "program_link": program_link,
            "category": category,
            "best_in_category": best_in_category
        })

    # Close the WebDriver
    driver.quit()

    return scraped_data


In [10]:
def main():
    # List to store all scraped data
    all_scraped_data = []

    # Loop through each page
    for page_number in range(1, 910):
        # URL for the current page
        page_url = f"https://www.naco.org/page/achievement-awards?_page={page_number}"
        logging.info(f"Scraping page {page_number}")

        # Scrape the awards information from the current page
        page_data = scrape_awards_table(page_url)
        all_scraped_data.extend(page_data)
        # Write the data to a CSV file after each page
        with open('naco_achievement_database.csv', mode='a', newline='', encoding='utf-8') as file:
            fieldnames = ["year", "county", "program", "program_link", "category", "best_in_category"]
            writer = csv.DictWriter(file, fieldnames=fieldnames)

            # Write header if the file is empty
            if file.tell() == 0:
                writer.writeheader()

            # Append each row of data to the CSV file
            for award in page_data:
                writer.writerow(award)
    logging.info("Scraping completed")

#     all_scraped_data_df = pd.DataFrame(all_scraped_data)
#     all_scraped_data_df.to_csv("naco_achievement_database.csv", index = False)

    # Print the total number of awards scraped
    logging.info("Total number of awards scraped:", len(all_scraped_data))

    # Print a sample of the scraped data
    logging.info("Sample of scraped data:")
    for award in all_scraped_data[:5]:  # Print the first 5 awards as a sample
        print(award)

In [11]:
start = datetime.now()
print(f'the download started at {str(start)[0:16]}')

the download started at 2024-06-08 14:08


In [12]:
# Run the main function
if __name__ == "__main__":
    main()

{'year': '2023', 'county': 'Citrus County, Fla.', 'program': 'Scripting for Security, Standardization, and Savings', 'program_link': 'naco.org/resources/award-programs/scripting-security-standardization-and-savings', 'category': 'Information Technology', 'best_in_category': False}
{'year': '2023', 'county': 'Montgomery County, Md.', 'program': 'Kids Day Out', 'program_link': 'naco.org/resources/award-programs/kids-day-out', 'category': 'Parks and Recreation', 'best_in_category': False}
{'year': '2023', 'county': 'Montgomery County, Md.', 'program': 'Let Her Shine – Girls Volleyball', 'program_link': 'naco.org/resources/award-programs/let-her-shine-%E2%80%93-girls-volleyball', 'category': 'Parks and Recreation', 'best_in_category': False}
{'year': '2023', 'county': 'Montgomery County, Md.', 'program': 'Inclusion Multi-Sport Program', 'program_link': 'naco.org/resources/award-programs/inclusion-multi-sport-program', 'category': 'Parks and Recreation', 'best_in_category': False}
{'year': 

--- Logging error ---
Traceback (most recent call last):
  File "/Users/kivan/opt/anaconda3/lib/python3.9/logging/__init__.py", line 1083, in emit
    msg = self.format(record)
  File "/Users/kivan/opt/anaconda3/lib/python3.9/logging/__init__.py", line 927, in format
    return fmt.format(record)
  File "/Users/kivan/opt/anaconda3/lib/python3.9/logging/__init__.py", line 663, in format
    record.message = record.getMessage()
  File "/Users/kivan/opt/anaconda3/lib/python3.9/logging/__init__.py", line 367, in getMessage
    msg = msg % self.args
TypeError: not all arguments converted during string formatting
Call stack:
  File "/Users/kivan/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/kivan/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/kivan/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 16, in <module>
    app.launch_

In [13]:
end = datetime.now()
total_time = end-start
print(f'the download ended at {str(end)[0:16]}')
print(f'the download took {str(total_time)[0:7]} to complete')

the download ended at 2024-06-08 19:12
the download took 5:04:06 to complete


In [14]:
import sys
import bs4
print(f'last updated: {datetime.now().strftime("%Y-%m-%d %H:%M")} \n')
print(f"System and module version information: \n")
print(f"Python version: {sys.version_info}")
print(f"pandas version: {pd.__version__}")
print(f"Beautiful Soup version: {bs4.__version__}")

last updated: 2024-06-08 19:12 

System and module version information: 

Python version: sys.version_info(major=3, minor=9, micro=7, releaselevel='final', serial=0)
pandas version: 2.0.3
Beautiful Soup version: 4.8.2
